In [257]:
# !pip install selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
import requests
import os
from io import BytesIO
# !pip install bs4
import time
from PIL import Image
from bs4 import BeautifulSoup
import base64
import matplotlib.pyplot as plt
from selenium.webdriver.common.keys import Keys

In [258]:
driver = webdriver.Chrome()
options = webdriver.ChromeOptions()
options.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)
options.add_argument("--disable-blink-features=AutomationControlled")

In [262]:
IMAGE_DEFAULT_PATH='./mockData/'
NUM=30
classes=[
     'BitterMelonSoup', 'BooPadPongali', 'Curried FishCake', 'Dumpling', 'EggsStewed', 
    'Fried Chicken', 'FriedKale', 'Fried Mussel Pancakes', 'Gaeng Jued', 'GaengKeawWan', 
    'GaiYang', 'GoongObWoonSen', 'GoongPao', 'GrilledQquid', 'HoyKraeng', 'HoyLaiPrikPao', 
    'Joke', 'KaiJeowMooSaap', 'KaiThoon', 'KaoManGai', 'KaoMooDang', 'KhanomJeenNamYaKati', 
    'KhaoMokGai', 'KhaoMooTodGratiem', 'KhaoNiewMaMuang', 'KkaoKlukkaphi', 'KorMooYang', 
    'Kuakling', 'KuayJab', 'Kuay TeowReua', 'LarbMoo', 'MassamanGai', 'MooSatay', 'Nam TokMoo', 
    'PadPakBung', 'PadPakRuamMit', 'PadThai', 'PadYordMala', 'PhatKaphrao', 'PorkStickyNoodles', 
    'Roast duck', 'Roast fish', 'Somtam', 'SoninLawEggs', 'Stewed PorkLeg', 'Suki', 
    'TomKhaGai', 'TomYum Goong', 'YamWoonSen', 'Yentafo'
]
thai_food_translation = {
    "BitterMelonSoup": "ต้มจืดมะระ",
    "BooPadPongali": "ปูผัดผงกะหรี่",
    "Curried FishCake": "ทอดมันปลา",
    "Dumpling": "เกี๊ยว",
    "EggsStewed": "ไข่พะโล้",

    "Fried Chicken": "ไก่ทอด",
    "FriedKale": "คะน้าผัด",
    "Fried Mussel Pancakes": "หอยทอด",
    "Gaeng Jued": "แกงจืด",
    "GaengKeawWan": "แกงเขียวหวาน",

    "GaiYang": "ไก่ย่าง",
    "GoongObWoonSen": "กุ้งอบวุ้นเส้น",
    "GoongPao": "กุ้งเผา",
    "GrilledQquid": "ปลาหมึกย่าง",
    "HoyKraeng": "หอยแครงลวก",

    "HoyLaiPrikPao": "หอยลายผัดพริกเผา",
    "Joke": "โจ๊ก",
    "KaiJeowMooSaap": "ไข่เจียวหมูสับ",
    "KaiThoon": "ไข่ตุ๋น",
    "KaoManGai": "ข้าวมันไก่",

    "KaoMooDang": "ข้าวหมูแดง",
    "KhanomJeenNamYaKati": "ขนมจีนน้ำยากะทิ",
    "KhaoMokGai": "ข้าวหมกไก่",
    "KhaoMooTodGratiem": "ข้าวหมูทอดกระเทียม",
    "KhaoNiewMaMuang": "ข้าวเหนียวมะม่วง",

    "KkaoKlukkaphi": "ข้าวคลุกกะปิ",
    "KorMooYang": "คอหมูย่าง",
    "Kuakling": "คั่วกลิ้ง",
    "KuayJab": "ก๋วยจั๊บ",
    "Kuay TeowReua": "ก๋วยเตี๋ยวเรือ",

    "LarbMoo": "ลาบหมู",
    "MassamanGai": "มัสมั่นไก่",
    "MooSatay": "หมูสะเต๊ะ",
    "Nam TokMoo": "น้ำตกหมู",
    "PadPakBung": "ผัดผักบุ้ง",

    "PadPakRuamMit": "ผัดผักรวมมิตร",
    "PadThai": "ผัดไทย",
    "PadYordMala": "ผัดยอดมะระ",
    "PhatKaphrao": "ผัดกะเพรา",
    "PorkStickyNoodles": "ราดหน้าหมู",

    "Roast duck": "เป็ดย่าง",
    "Roast fish": "ปลาเผา",
    "Somtam": "ส้มตำ",
    "SoninLawEggs": "ไข่ลูกเขย",
    "Stewed PorkLeg": "ขาหมูพะโล้",

    "Suki": "สุกี้",
    "TomKhaGai": "ต้มข่าไก่",
    "TomYum Goong": "ต้มยำกุ้ง",
    "YamWoonSen": "ยำวุ้นเส้น",
    "Yentafo": "เย็นตาโฟ"
}

for c in classes:
    os.makedirs(os.path.join(IMAGE_DEFAULT_PATH, c), exist_ok=True)

In [260]:
def go(query):
    driver.get("https://images.google.com/")

    search = driver.find_element(By.CSS_SELECTOR, "textarea")
    search.send_keys(query)
    search.send_keys(Keys.ENTER)

    time.sleep(3)

    images = driver.find_elements(By.CSS_SELECTOR, ".H8Rx8c img")

    src_list = []
    for img in images:
        src = img.get_attribute("src") or img.get_attribute("data-src")

        if src and (src.startswith("http") or src.startswith("data:image")):
            src_list.append(src)

        if len(src_list) >= NUM:
            break

    return src_list

def go_get(query):
    driver.get("https://images.google.com/")

    search = driver.find_element(By.CSS_SELECTOR, "textarea")
    search.send_keys(query)
    search.send_keys(Keys.ENTER)

    time.sleep(3)

    thumbnails = driver.find_elements(By.CSS_SELECTOR, ".H8Rx8c img")

    src_list = []

    for thumb in thumbnails:

        try:
            thumb.click()
            time.sleep(1)

            images = driver.find_elements(By.CSS_SELECTOR, "img.sFlh5c")
            # print(images)
            for img in images:
                src = img.get_attribute("src")

                if src and src.startswith("http"):
                    src_list.append(src)
                    break

            if len(src_list) >= NUM:
                break

        except:
            continue

    return src_list
def open_image(data):

    if data.startswith("data:image"):   # base64 image
        header, encoded = data.split(",", 1)
        img_bytes = base64.b64decode(encoded)

    else:   # normal URL
        img_bytes = requests.get(data).content

    image = Image.open(BytesIO(img_bytes))

    plt.imshow(image)

def save_image(url, filename, class_name):
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()

        with open(f"{IMAGE_DEFAULT_PATH}/{class_name}/{filename}", "wb") as f:
            f.write(r.content)

        print("saved:", filename)

    except Exception as e:
        print("failed:", url, e)

def save_image_size_limit(url, filename, class_name, min_size=500):
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()

        img = Image.open(BytesIO(r.content))
        w, h = img.size

        # skip small images
        if w < min_size or h < min_size:
            print("skip small:", w, h)
            return False

        path = f"{IMAGE_DEFAULT_PATH}/{class_name}/{filename}"
        img.save(path)

        print("saved:", filename, w, h)
        return True

    except Exception as e:
        print("failed:", url, e)
        return False
def save_all_size_limit(contents, class_name):

    path = f"{IMAGE_DEFAULT_PATH}/{class_name}/"
    os.makedirs(path, exist_ok=True)

    # clear folder
    for file in os.listdir(path):
        file_path = os.path.join(path, file)
        if os.path.isfile(file_path):
            os.remove(file_path)

    saved = 0

    for url in contents:
        filename = f"{saved+1}.jpg"

        ok = save_image_size_limit(url, filename, class_name, 500)

        if ok:
            saved += 1

        if saved >= NUM:
            break

def save_all(contents, class_name):
    path = f"{IMAGE_DEFAULT_PATH}/{class_name}/"
    os.makedirs(path, exist_ok=True)
    for file in os.listdir(path):
        file_path = os.path.join(path, file)
        if os.path.isfile(file_path):
            os.remove(file_path)
    for i,url in enumerate(contents):
        filename = f"{i+1}.jpg"
        save_image(url, filename, class_name)
def fix_recap():
    input(f"Press ENTER after solving CAPTCHA")
def go_get_save(query, class_name):

    driver.get("https://images.google.com/")

    search = driver.find_element(By.CSS_SELECTOR, "textarea")
    search.clear()
    search.send_keys(query)
    search.send_keys(Keys.ENTER)

    time.sleep(3)

    path = f"{IMAGE_DEFAULT_PATH}/{class_name}/"
    os.makedirs(path, exist_ok=True)

    # clear old images
    for f in os.listdir(path):
        os.remove(os.path.join(path, f))

    # thumbnails = driver.find_elements(By.CSS_SELECTOR, ".H8Rx8c img")

    saved = 0
    visited = set()

    i = 0
    
    while saved < NUM:
        thumbnails = driver.find_elements(By.CSS_SELECTOR, ".H8Rx8c img")
        if i >= len(thumbnails):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            buttons1 = driver.find_elements(By.CSS_SELECTOR, ".JVy8Bc a")
            buttons2 = driver.find_elements(By.CSS_SELECTOR, "a.T7sFge")
            if buttons1:
                buttons1[0].click()
                time.sleep(2)
                continue

            if buttons2:
                buttons2[0].click()
                time.sleep(2)
                continue
            time.sleep(2)
            thumbnails = driver.find_elements(By.CSS_SELECTOR, ".H8Rx8c img")

            # still no more images → stop
            if i >= len(thumbnails):
                
                print("No more images available")
                break
        thumb = thumbnails[i]
        i += 1

        try:
            thumb.click()
            time.sleep(1)

            images = driver.find_elements(By.CSS_SELECTOR, "img.sFlh5c")

            for img in images:
                src = img.get_attribute("src")

                if not src or not src.startswith("http"):
                    continue

                if src in visited:
                    continue

                visited.add(src)

                filename = f"{class_name}_{saved+1}.jpg"

                ok = save_image_size_limit(src, filename, class_name, 400)

                if ok:
                    saved += 1
                    break

        except:
            continue

    print("Total saved:", saved)
def get_all_image_class():
    fix_recap()
    for c in classes:
        go_get_save(f"{thai_food_translation[c]}", c)
        # save_all(content, c)

In [261]:

get_all_image_class()

skip small: 225 225
saved: Somtam_20.jpg 2560 1707
skip small: 275 183
failed: https://www.gourmetandcuisine.com/Images/editor_upload/_editor20190225113640_original.jpg 403 Client Error: Forbidden for url: https://www.gourmetandcuisine.com/Images/editor_upload/_editor20190225113640_original.jpg
skip small: 275 183
skip small: 275 183
saved: Somtam_21.jpg 1000 667
saved: Somtam_22.jpg 1280 720
skip small: 300 168
failed: https://img.wongnai.com/p/1968x0/2022/08/13/2702cbb8d3f546e5b6d2602927e02f91.jpg 403 Client Error: Forbidden for url: https://img.wongnai.com/p/1968x0/2022/08/13/2702cbb8d3f546e5b6d2602927e02f91.jpg
skip small: 217 233
skip small: 275 183
saved: Somtam_23.jpg 1200 800
skip small: 686 386
skip small: 299 168
skip small: 188 269
skip small: 277 182
saved: Somtam_24.jpg 960 633
saved: Somtam_25.jpg 960 720
skip small: 259 194
failed: https://www.beartai.com/wp-content/uploads/2023/04/%E0%B8%9A%E0%B8%97%E0%B8%84%E0%B8%A7%E0%B8%B2%E0%B8%A1-brief-%E0%B8%AA%E0%B9%89%E0%B8%A1%E

In [263]:
rerun_list = ["Roast fish", "PorkStickyNoodles"]
for c in rerun_list:
    go_get_save(f"{thai_food_translation[c]}", c)

failed: https://img.wongnai.com/p/1920x0/2020/05/06/c0dedcc16127483f9eace9973b040ebc.jpg 403 Client Error: Forbidden for url: https://img.wongnai.com/p/1920x0/2020/05/06/c0dedcc16127483f9eace9973b040ebc.jpg
skip small: 259 194
skip small: 225 225
skip small: 275 183
skip small: 290 174
skip small: 310 163
saved: Roast fish_1.jpg 1200 630
skip small: 225 225
skip small: 194 259
skip small: 299 168
skip small: 686 386
skip small: 270 187
saved: Roast fish_2.jpg 1540 1064
saved: Roast fish_3.jpg 670 402
skip small: 290 174
saved: Roast fish_4.jpg 640 640
skip small: 225 225
saved: Roast fish_5.jpg 680 781
skip small: 209 241
saved: Roast fish_6.jpg 960 633
skip small: 277 182
saved: Roast fish_7.jpg 670 402
skip small: 290 174
skip small: 686 386
skip small: 299 168
skip small: 260 194
failed: https://img.wongnai.com/p/1920x0/2020/12/08/62e66b5c07d24e9d944a1bc0ab668a23.jpg 403 Client Error: Forbidden for url: https://img.wongnai.com/p/1920x0/2020/12/08/62e66b5c07d24e9d944a1bc0ab668a23.jpg

'<div class="ZFiwCf"><span class="LGwnxb JGD2rd">ดูเพิ่ม</span><span class="w2fKdd z1asCe" style="height:20px;line-height:20px;width:20px"><svg focusable="false" aria-hidden="true" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M8.59 16.59L13.17 12 8.59 7.41 10 6l6 6-6 6-1.41-1.41z"></path></svg></span></div>'